# CardioTwin - Model Training (Estudiante B - Día 1)
Este notebook aisla el trabajo de Machine Learning. Se encarga de:
1. Cargar el dataset de Framingham.
2. Limpiar nulos.
3. Aplicar SMOTE para manejar el desbalance (CHD ~15%).
4. Entrenar XGBoost.
5. Evaluar con AUC-ROC.
6. Serializar el modelo con joblib.
7. Integrar SHAP y validar.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

# 1. Cargar Framingham
df = pd.read_csv('../../../data/framingham.csv')
print("Shape original:", df.shape)
df.head()

Shape original: (4240, 16)


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [3]:
# 2. Limpiar nulos
df = df.dropna()
print("Shape sin nulos:", df.shape)

# Separar features (X) y target (y)
X = df.drop(columns=['TenYearCHD'])
y = df['TenYearCHD']

# Verificar prevalencia
print("\nPrevalencia CHD:")
print(y.value_counts(normalize=True))

Shape sin nulos: (3658, 16)

Prevalencia CHD:
TenYearCHD
0    0.847731
1    0.152269
Name: proportion, dtype: float64


In [4]:
# Split de datos (antes de SMOTE para evitar data leakage en testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Aplicar SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Shape tras SMOTE:", X_train_resampled.shape)
print("Prevalencia en train resampled:")
print(y_train_resampled.value_counts(normalize=True))

Shape tras SMOTE: (4960, 15)
Prevalencia en train resampled:
TenYearCHD
0    0.5
1    0.5
Name: proportion, dtype: float64


In [5]:
# 4. Entrenar XGBoost
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train_resampled, y_train_resampled)


/Users/juanleal/Desktop/Cardiotwin/.venv/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [11:28:45] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [6]:
# 5. Evaluar con AUC-ROC
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

AUC-ROC: 0.6258

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.92      0.89       621
           1       0.26      0.16      0.20       111

    accuracy                           0.80       732
   macro avg       0.56      0.54      0.54       732
weighted avg       0.77      0.80      0.78       732



In [7]:
# 6. Serializar el modelo con joblib
joblib.dump(model, '../xgboost_chd_model.joblib')
print("Modelo serializado en '../xgboost_chd_model.joblib'")

Modelo serializado en '../xgboost_chd_model.joblib'


In [8]:
# 7. Integrar shap.TreeExplainer y validar
explainer = shap.TreeExplainer(model)

# Tomar un registro de ejemplo
sample = X_test.iloc[[0]]
shap_values = explainer.shap_values(sample)

print("Valores SHAP para el registro de ejemplo:")
print(shap_values)

# Mostrar importancia base de features (opcional en Jupyter, descomentar si usas la UI)
# shap.summary_plot(explainer.shap_values(X_test), X_test, plot_type="bar")

Valores SHAP para el registro de ejemplo:
[[ 0.09150507  0.25460523 -0.9964063  -0.2761324   0.09245741 -0.11196208
   0.         -0.21697047  0.0134212   0.60018235 -0.54519314 -0.06148759
  -0.311009   -0.13061994 -0.23069532]]
